# PPO Metric Reosurce Managment

In [ ]:
import os

from stable_baselines3.common.callbacks import CallbackList

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback
import logging

In [ ]:
total_timesteps = 500_000

policy_kwargs = dict(
    net_arch=[128,64,64]
)

config = {
    "policy_type": "MultiInputPolicy",
    "total_timesteps": total_timesteps,
    "env_name": "ClusterScheduling-metric-offline-v1",
}

In [ ]:
run = wandb.init(
    project="cluster-scheduling",
    config=config,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/guyarieli/.netrc.
wandb: Currently logged in as: guyarieli17 (dev0guy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import os

from stable_baselines3.common.callbacks import CallbackList
from src.experiment.callbacks.metrics import CustomMetricsCallback
from src.experiment.common.models.metric_cnn import DeepRMCNNExtractor
from src.experiment.common.wrappers_new.job_zero_by_status import ZeroJobsByStatusWrapper
from src.server_simulator.envs.cluster_simulator.base.extractors.reward import AverageSlowDownReward



os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"


import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecVideoRecorder
import wandb
from wandb.integration.sb3 import WandbCallback
import logging
logging.basicConfig(level=logging.ERROR)
from src.experiment.common.wrappers import FlattenActionWrapper, TimeLimitPenaltyWrapper

def main():
    policy_kwargs = dict(
        features_extractor_class=DeepRMCNNExtractor,
        net_arch=[64, 32, 64]
    )

    config = {
        "policy_type": "MultiInputPolicy", # MultiInputPolicy
        "total_timesteps": 300_000,
        "env_name": "ClusterScheduling-metric-offline-v1",
    }

    run = wandb.init(
        project="cluster-scheduling",
        config=config,
        sync_tensorboard=True,
        monitor_gym=True,
        save_code=True,
    )


    def make_env():
        n_jobs = 5
        n_machines = 1
        n_resources = 2
        n_ticks = 10
        max_episode_steps = 100
        penalty = -1e4
        print(f"{n_jobs=}, {n_machines=}, {n_resources=}, {n_ticks=}, {max_episode_steps=}, {penalty=}")
        reward_caculator=AverageSlowDownReward(n_jobs)
        env = gym.make(
            config["env_name"],
            render_mode="rgb_array",
            n_jobs=n_jobs,
            n_machines=n_machines,
            n_resources=n_resources,
            n_ticks=n_ticks,
            reward_caculator=reward_caculator
        )
        env = TimeLimitPenaltyWrapper(env, max_episode_steps=max_episode_steps, penalty=penalty)
        env = ZeroJobsByStatusWrapper(env)
        env = FlattenActionWrapper(env)
        env = Monitor(env)
        return env

    env = DummyVecEnv([make_env])
    env = VecVideoRecorder(
        env,
        f"videos/{run.id}",
        record_video_trigger=lambda x: x % 2_000 == 0,
        video_length=200,
    )
    model = PPO(
        config["policy_type"],
        env,
        policy_kwargs=policy_kwargs,
        verbose=1,
        tensorboard_log=f"runs/{run.id}"
    )
    wandb_callback = WandbCallback(
        gradient_save_freq=2_000,
        model_save_path=f"models/{run.id}",
        verbose=2,
    )
    metric_callback = CustomMetricsCallback(verbose=1)
    model.learn(
        total_timesteps=config["total_timesteps"],
        callback=CallbackList([
            metric_callback,
            wandb_callback
        ]),
    )


if __name__ == '__main__':
    main()